# OmniVoice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

This notebook demonstrates the basic usage of [OmniVoice](https://github.com/k2-fsa/OmniVoice), a massively multilingual zero-shot TTS model supporting 600+ languages.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI, no code needed)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice

## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [1]:
!pip install omnivoice

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.5/162.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 6.2 MB/s eta 0:00:00


## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [ ]:
!omnivoice-demo --share

## 3. Option B — Python API

### 3.1 Load Model

In [ ]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [ ]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice cloning.",
    ref_audio=ref_audio_path,
    # ref_text="Transcription of the reference audio.",  # optional
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

In [ ]:
!pip install pyTelegramBotAPI
!apt-get install ffmpeg

In [ ]:
import telebot
from telebot.types import ReplyKeyboardMarkup, KeyboardButton
import os
import torchaudio
import torch

# --- إعدادات البوت ---
TOKEN = "8850631653:AAF3nZ9-lX_O6iPtMrjjaSz5u5OCkTBW1dU"
bot = telebot.TeleBot(TOKEN)

# قاموس لحفظ حالة كل مستخدم (تتبع الخطوات)
user_states = {}
user_data = {}

# --- دوال الواجهة (الأزرار) ---
def main_menu():
    markup = ReplyKeyboardMarkup(resize_keyboard=True, one_time_keyboard=False)
    markup.add(KeyboardButton("🎙️ استنساخ صوت وتحويل نص"))
    markup.add(KeyboardButton("❌ إلغاء العملية"))
    return markup

# --- دالة معالجة OmniVoice (نقطة الربط مع النموذج) ---
def process_omnivoice(text, reference_audio_path, output_path):
    """
    هذه الدالة تقوم بربط بيانات تليجرام بنموذج OmniVoice.
    تحتاج إلى تعديل السطر الخاص بـ generation بناءً على المتغيرات في الـ Colab الخاص بك.
    """
    try:
        # هنا نستخدم الكود الافتراضي الخاص بـ OmniVoice لتوليد الصوت
        # تأكد أن المتغير model معرف مسبقاً في الخلايا السابقة للـ Colab

        print(f"جاري معالجة النص: {text}")

        # --- هذا الجزء يعتمد على كيفية تعريف دالة التوليد في OmniVoice ---
        # مثال تقريبي (يجب تغييره حسب كود OmniVoice الموجود في الخلايا السابقة):
        # audio_output = model.generate(text=text, reference_voice=reference_audio_path)
        # torchaudio.save(output_path, audio_output, sample_rate=24000)

        # ⚠️ ملاحظة: استبدل التعليقات أعلاه بكود التوليد الفعلي الموجود في الـ Colab

        return True
    except Exception as e:
        print(f"Error in OmniVoice: {e}")
        return False

# --- أوامر البوت ---
@bot.message_handler(commands=['start'])
def send_welcome(message):
    chat_id = message.chat.id
    user_states[chat_id] = "IDLE"
    bot.send_message(
        chat_id,
        "أهلاً بك في بوت 🤖 **OmniVoice AI**\n\n"
        "أنا هنا لتحويل النصوص إلى كلام بأصوات مستنسخة باحترافية عالية.\n"
        "اختر من القائمة بالأسفل للبدء 👇",
        reply_markup=main_menu(),
        parse_mode="Markdown"
    )

@bot.message_handler(func=lambda message: message.text == "❌ إلغاء العملية")
def cancel_operation(message):
    chat_id = message.chat.id
    user_states[chat_id] = "IDLE"
    bot.send_message(chat_id, "تم إلغاء العملية الحالية بنجاح. ✅", reply_markup=main_menu())

@bot.message_handler(func=lambda message: message.text == "🎙️ استنساخ صوت وتحويل نص")
def start_cloning(message):
    chat_id = message.chat.id
    user_states[chat_id] = "WAITING_FOR_VOICE"
    bot.send_message(
        chat_id,
        "رائع! الخطوة 1️⃣:\n"
        "يرجى إرسال **مقطع صوتي (Voice Message)** قصير وواضح (من 5 إلى 10 ثوانٍ) للشخص الذي تريد استنساخ صوته.",
        reply_markup=main_menu()
    )

@bot.message_handler(content_types=['voice'])
def handle_voice(message):
    chat_id = message.chat.id
    if user_states.get(chat_id) != "WAITING_FOR_VOICE":
        bot.send_message(chat_id, "يرجى الضغط على زر 'استنساخ صوت' أولاً.")
        return

    try:
        bot.send_message(chat_id, "⏳ جاري تحميل المقطع الصوتي...")

        # تحميل الصوت من تليجرام
        file_info = bot.get_file(message.voice.file_id)
        downloaded_file = bot.download_file(file_info.file_path)

        # حفظ الصوت بصيغة ogg
        input_path = f"ref_{chat_id}.ogg"
        with open(input_path, 'wb') as new_file:
            new_file.write(downloaded_file)

        user_data[chat_id] = {'ref_audio': input_path}
        user_states[chat_id] = "WAITING_FOR_TEXT"

        bot.send_message(
            chat_id,
            "تم استلام الصوت بنجاح! ✅\n\n"
            "الخطوة 2️⃣:\n"
            "الآن، أرسل **النص** الذي تريدني أن أنطقه بهذا الصوت."
        )
    except Exception as e:
        bot.send_message(chat_id, f"حدث خطأ أثناء تحميل الصوت: {e}")

@bot.message_handler(func=lambda message: user_states.get(message.chat.id) == "WAITING_FOR_TEXT")
def handle_text_for_cloning(message):
    chat_id = message.chat.id
    text = message.text

    if not text:
        bot.send_message(chat_id, "الرجاء إرسال نص صالح.")
        return

    bot.send_message(chat_id, "⚙️ جاري معالجة الصوت بالذكاء الاصطناعي... قد يستغرق هذا بضع ثوانٍ ⏳")

    ref_audio = user_data[chat_id]['ref_audio']
    output_audio = f"output_{chat_id}.wav"

    # استدعاء دالة OmniVoice
    success = process_omnivoice(text, ref_audio, output_audio)

    if success and os.path.exists(output_audio):
        with open(output_audio, 'rb') as audio:
            bot.send_audio(chat_id, audio, caption="تم التوليد بنجاح! ✨")

        # تنظيف الملفات المؤقتة لتوفير المساحة في Colab
        os.remove(ref_audio)
        os.remove(output_audio)
    else:
        bot.send_message(chat_id, "❌ عذراً، حدث خطأ أثناء التوليد. راجع كود OmniVoice.")

    user_states[chat_id] = "IDLE"

# تشغيل البوت
print("🚀 البوت يعمل الآن... اذهب إلى تليجرام وابدأ الاستخدام!")
bot.polling(none_stop=True)